# ATENA-TF Master Welcome — Google Colab Edition

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Eden-Mironi/Atena-TF/blob/main/Notebooks/ATENA-TF-Master-Welcome-Colab.ipynb)

**One-click start:** Click **Runtime → Run all** and wait ~2 min for setup, then explore!

This notebook is the exact port of ATENA-Master's Welcome notebook to TensorFlow 2.
It demonstrates the environment, reward system, and custom action sequences — no training required.

---


In [1]:
# ── Colab / Local Setup ──────────────────────────────────────────────────────
# This cell detects whether we are in Google Colab and, if so:
#   1. Installs all required packages
#   2. Clones the ATENA-TF repository from GitHub
#   3. Adds the project root to sys.path
# If running locally (Jupyter), nothing extra happens — just run the cell.
import sys, os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import subprocess

    print("📦 Installing dependencies (~2 min on first run)…")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "tensorflow>=2.16.0", "tensorflow-probability>=0.24.0"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "gym==0.25.2", "numpy<2.0.0", "pandas", "matplotlib",
        "seaborn", "scipy", "cachetools", "zss", "nltk", "openpyxl"])
    print("✅ Dependencies installed!")

    if not os.path.exists("/content/atena-tf"):
        print("📥 Cloning ATENA-TF from GitHub…")
        subprocess.check_call(["git", "clone",
            "https://github.com/Eden-Mironi/Atena-TF.git",
            "/content/atena-tf"])
        print("✅ Repository cloned!")

    os.chdir("/content/atena-tf")

    if "/content/atena-tf" not in sys.path:
        sys.path.insert(0, "/content/atena-tf")

    print("✅ ATENA-TF project ready!")

print(f"🌍 Environment : {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print(f"📁 Working dir : {os.getcwd()}")


📦 Installing dependencies (~2 min on first run)…
✅ Dependencies installed!
📥 Cloning ATENA-TF from GitHub…
✅ Repository cloned!


CalledProcessError: Command '['/usr/bin/python3', '-m', 'pip', 'install', '-q', '-e', '.']' returned non-zero exit status 1.

# ATENA-TF Master Welcome

## Exact Port of ATENA-Master Welcome Notebook to TensorFlow 2

This notebook replicates the exact structure of the original `ATENA- Welcome.ipynb` from ATENA-master,
but uses the TensorFlow 2 framework components.

### 📂 Important: Working Directory

**To run this notebook successfully**, start Jupyter from the project root directory:

```bash
# From the atena-tf 2 directory (NOT from Notebooks/)
cd /path/to/atena-tf-2
jupyter notebook
# Then open: Notebooks/ATENA-TF-Master-Welcome.ipynb
```

Alternatively, if running from within Notebooks/, the path setup in the next cell will automatically adjust.

### What you'll learn:
- Environment properties and setup
- Running random "agent" exploration
- Executing custom action sequences
- Understanding the action space and observations
- Viewing reward components and analysis
- Analyst view for detailed exploration


In [ ]:
# Fix for gym 0.25.2 compatibility with newer numpy
import numpy as np
if not hasattr(np, "bool8"):
    np.bool8 = np.bool_

# Setup paths - ensure we can import from parent directory
import sys
import os

# Get the notebook's directory and add parent to path
notebook_dir = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
parent_dir = os.path.dirname(notebook_dir) if "Notebooks" in notebook_dir else notebook_dir

# Add to path if not already there
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

# Core imports
import scipy as sp
import gym

# ATENA imports
import Configuration.config as cfg
from gym_atena.envs.atena_env_cont import ATENAEnvCont
from Evaluation.notebook_utils import *

%matplotlib inline

print("✅ All imports successful!")
print(f"📊 Working directory: {os.getcwd()}")
print(f"🎯 Schema: {cfg.schema}")
print(f"📈 Max steps per episode: {cfg.MAX_NUM_OF_STEPS}")


## Environment Setup

Create the ATENA environment. The environment will automatically load the Snorkel model for humanity scoring.


In [ ]:
%%capture
env = gym.make(env_d)


## Environment Properties

Let's explore the action and observation spaces:


In [ ]:
# Print environment properties
print("action space size is: ", env.action_space)
print("action space low is: ", env.action_space.low)
print("action space high is: ", env.action_space.high)
print("observation space dimensions are: ", env.observation_space)
print("observation space dimensions are: ", env.observation_space.low)
print("observation space dimensions are: ", env.observation_space.high)
print("reward range is:", env.reward_range)


## Reset Environment

Reset to the first dataset and view the initial observation:


In [ ]:
print(env.reset())
print("dataset number %d" % env.dataset_number)


# Run a Random "Agent"

Let's run a random agent that takes random actions in the environment:


In [ ]:
info_hist, r_sum = run_episode(env=env, compressed=True)
simulate(info_hist)


# Run Custom Actions

Now let's run a specific sequence of actions on dataset 0 (Flights dataset).

This example explores large flight delays from DFW airport:
1. Group by departure_delay
2. Filter to LARGE_DELAY
3. Group by origin_airport
4. Filter to DFW
5. Group by delay_reason
6. Group by scheduled_departure
7-12. Navigate back through the states


In [ ]:
# Set dataset_number
dataset_number = 0

# Set actions
action_vecs = [
[2, 6, 1, 0.3221167325973511, 0, 0],      #  1: Group by departure_delay
[1, 6, 1, -0.48, 0, 0],                   #  2: Filter LARGE_DELAY
[2, 2, 7, 0.48046758365631, 0, 0],        #  3: Group by origin_airport
[1, 2, 1, 0.48086163997650146, 0, 0],     #  4: Filter DFW
[2, 5, 8, -0.48018680667877, 0, 0],       #  5: Group by delay_reason
[2, 8, 8, 0.444186806678772, 0, 0],       #  6: Group by scheduled_departure
[0, 7, 4, 0.6415030360221863, 0, 0],      #  7: Back
[0, 5, 4, 0.8115951418876648, 0, 0],      #  8: Back
[2, 11, 1, -0.48, 0, 0],                  #  9: Group by day_of_year
[0, 5, 1, -0.45, 0, 0],                   # 10: Back
[0, 11, 4, 0.6415030360221863, 0, 0],     # 11: Back
[0, 6, 1, 0.5015030360221863, 0, 0],      # 12: Back
]

info_hist, r_sum = run_episode(
    dataset_number=dataset_number,
    env=env,
    compressed=False,
    filter_by_field=True,
    continuous_filter_term=True,
    actions_lst=action_vecs,
)

simulate(info_hist, displays=True)


## Understanding the Output

For each action, you'll see:
- **Raw action vector**: `[action_type, param1, param2, param3, param4, param5]`
- **Action description**: Human-readable action
- **Reward**: Total reward for this step
- **Reward components**: Breakdown of reward
  - `empty_display`: Penalty for empty results
  - `empty_groupings`: Penalty for empty groups
  - `same_display_seen_already`: Penalty for redundant actions
  - `back`: Reward for back actions
  - `diversity`: Reward for diverse exploration
  - `interestingness`: Reward for interesting patterns
  - `kl_distance`: KL divergence measure
  - `compaction_gain`: Data compaction reward
  - `humanity`: Rule-based humanity score
  - `snorkel_humanity`: ML-based humanity score
- **Human rules triggered**: Which human behavior rules matched
- **Snorkel rules triggered**: Which ML rules matched
- **Resulting DataFrame**: The data after applying the action


# Run Another Custom Action Sequence

Let's explore a different dataset (dataset 1):


In [ ]:
# Set dataset_number
dataset_number = 1

# Set actions
action_vecs = [
[2, 3, 0, 0.0, 0, 0],                     #  1: Group action
[1, 4, 8, -0.462690144777298, 0, 0],      #  2: Filter action
[0, 0, 0, 0.0, 0, 0],                     #  3: Back
[0, 0, 0, 0.0, 0, 0],                     #  4: Back
[1, 5, 2, -0.48248493671417236, 0, 0],    #  5: Filter action
[0, 0, 0, 0.0, 0, 0],                     #  6: Back
[2, 2, 0, 0.0, 0, 0],                     #  7: Group action
[2, 6, 0, 0.0, 0, 0],                     #  8: Group action
[2, 5, 0, 0.0, 0, 0],                     #  9: Group action
[1, 4, 8, -0.4228319823741913, 0, 0],     # 10: Filter action
[0, 0, 0, 0.0, 0, 0],                     # 11: Back
[1, 4, 8, -0.41552573442459106, 0, 0],    # 12: Filter action
]

info_hist, r_sum = run_episode(
    dataset_number=dataset_number,
    env=env,
    compressed=False,
    filter_by_field=True,
    continuous_filter_term=True,
    actions_lst=action_vecs,
)

simulate(info_hist, displays=True)


# Analyst View

The Analyst View provides a richer, more detailed display of the exploration session:
- HTML-formatted action descriptions
- Navigation buttons to jump between states
- Visual tree of the exploration path
- Current filtering state display
- Interactive dataframe displays


In [ ]:
# Set dataset_number
dataset_number = 3

# Set actions
action_vecs = [
[2, 6, 0, 0.0, 0, 0],                     #  1: Group action
[2, 5, 0, 0.0, 0, 0],                     #  2: Group action
[1, 4, 8, -0.49928829073905945, 0, 0],    #  3: Filter action
[0, 0, 0, 0.0, 0, 0],                     #  4: Back
[0, 0, 0, 0.0, 0, 0],                     #  5: Back
[1, 4, 8, -0.4994085431098938, 0, 0],     #  6: Filter action
[0, 0, 0, 0.0, 0, 0],                     #  7: Back
[0, 0, 0, 0.0, 0, 0],                     #  8: Back
[2, 3, 0, 0.0, 0, 0],                     #  9: Group action
[2, 2, 0, 0.0, 0, 0],                     # 10: Group action
[2, 10, 0, 0.0, 0, 0],                    # 11: Group action
[1, 3, 8, -0.4997364282608032, 0, 0],     # 12: Filter action
]

info_hist, r_sum = run_episode_analyst_view(
    dataset_number=dataset_number,
    env=env,
    compressed=False,
    filter_by_field=True,
    continuous_filter_term=True,
    actions_lst=action_vecs,
)


## Action Vector Format

Each action is a 6-element vector: `[action_type, param1, param2, param3, param4, param5]`

### Action Types:
- `0` = Back action
- `1` = Filter action
- `2` = Group action

### For Filter Actions:
- `param1` = column index
- `param2` = condition (eq, neq, gt, lt, etc.)
- `param3` = term (continuous value mapped to actual filter term)

### For Group Actions:
- `param1` = group column index
- `param2` = aggregation column index
- `param3` = aggregation function (len, sum, mean, etc.)

### For Back Actions:
- All params are typically 0 or don't matter


## Next Steps

Now that you understand the basics, you can:

1. **Explore other notebooks:**
   - `ATENA-TF-Welcome.ipynb` - Modern introduction with trained agents
   - `Live_Recommendations_System.ipynb` - Interactive recommender
   - `Master_Compatible_Evaluation.ipynb` - Comprehensive evaluation
   - `evaluate_agent_notebook.ipynb` - Agent performance analysis

2. **Train your own agent:**
   ```bash
   python main.py
   ```

3. **Modify configurations:**
   - Edit `Configuration/config.py` to change:
     - Reward coefficients
     - Dataset schema
     - Episode length
     - Training hyperparameters

4. **Create custom action sequences:**
   - Use the action vector format above
   - Test different exploration strategies
   - Compare rewards across different approaches

5. **Analyze reward components:**
   - Study which actions get high rewards
   - Understand the humanity scoring
   - Optimize for different reward components


## Key Differences from Original ATENA-Master

While this notebook replicates the original structure, there are some key differences in the TF2 version:

1. **Backend**: Uses TensorFlow 2 instead of ChainerRL
2. **Algorithm**: PPO implementation instead of ChainerRL's PPO
3. **Environment**: Enhanced environment with better logging and tracking
4. **Snorkel**: Updated Snorkel compatibility layer for newer Python/libraries
5. **Reward System**: Improved reward stabilization and coefficient handling
6. **Evaluation**: More comprehensive evaluation metrics and tools

The core ATENA concepts, action space, and reward structure remain the same!
